In [3]:
!pip install -q langchain==0.1.16 langchain-community==0.0.34 faiss-cpu pypdf

In [4]:
from google.colab import files
files.upload()

{}

In [5]:
import os, zipfile

if "compressed_resume.zip" in os.listdir():
    with zipfile.ZipFile("compressed_resume.zip", 'r') as zip_ref:
        zip_ref.extractall("data")
    folder = "data"
else:
    folder = "."

In [6]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader

documents = []

for file in os.listdir(folder):
    path = os.path.join(folder, file)

    if file.endswith(".txt"):
        documents.extend(TextLoader(path).load())

    elif file.endswith(".pdf"):
        documents.extend(PyPDFLoader(path).load())

print("✅ Documents Loaded:", len(documents))

✅ Documents Loaded: 1


In [8]:
!pip install pytesseract pdf2image
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [11]:
!	apt-get install -y poppler-utils
from pdf2image import convert_from_path
import pytesseract

def ocr_pdf(file_path):
    pages = convert_from_path(file_path)
    text = ""

    for page in pages:
        text += pytesseract.image_to_string(page)

    return text

# Run OCR
ocr_text = ocr_pdf("compressed_resume.pdf")

print(ocr_text[:500])  # preview

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 6 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (193 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
RESUME

Subhiksha N

57/21 Bharathi Street, Veerappan Chatram,erode

Erode

Tamil Nadu - 638004

Mob No, : 7904154627 .

Email td : subibcloud1108@gmail.com |

CAREER OBJECTIVE

To make 

In [13]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document # Import Document class

# Create a Document from the OCR text
# Assuming ocr_text is available from the previous cell's execution
doc_from_ocr = Document(page_content=ocr_text, metadata=documents[0].metadata if documents else {})

# Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = splitter.split_documents([doc_from_ocr]) # Pass a list containing the new document

# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Vector Store
vector_db = FAISS.from_documents(docs, embeddings)

print("✅ Document Integration Done")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Document Integration Done


In [14]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

def retrieve(query):
    return retriever.invoke(query)

In [15]:
import re

def generate(query, docs):
    context = " ".join([d.page_content for d in docs])
    q = query.lower()

    # 🔹 Skills
    if "skill" in q:
        match = re.search(r"(C.*JavaScript)", context)
        return match.group(1) if match else "Skills not found"

    # 🔹 Career Objective
    elif "career objective" in q:
        match = re.search(r"CAREER OBJECTIVE(.*?)(ACADEMIC|OTHER)", context, re.DOTALL)
        return match.group(1).strip() if match else "Career objective not found"

    # 🔹 Education
    elif "qualification" in q or "education" in q:
        match = re.search(r"(BTech.*?CGPA)", context)
        return match.group(1) if match else "Education not found"

    # 🔹 Name
    elif "name" in q:
        match = re.search(r"RESUME\s+([A-Za-z ]+)", context)
        return match.group(1) if match else "Name not found"

    # 🔹 Languages
    elif "language" in q:
        match = re.search(r"Language Known\s*:\s*(.*)", context)
        return match.group(1) if match else "Languages not found"

    # 🔹 Summary
    elif "summary" in q:
        return context[:300] + "..."

    # 🔹 Default
    return "Answer not found clearly in document"

In [24]:
query = input("Ask your question: ")

docs = retrieve(query)
answer = generate(query, docs)

print("\n🤖 Answer:\n", answer)

Ask your question: what is the name of the person?

🤖 Answer:
 Subhiksha N
